# H2S-NN Repeated 10 x 10 K-Fold Graph Classification

This notebook runs 10 repetitions of 10-fold KFold evaluation. Each repetition/fold pair has an independent W&B Bayesian/Hyperband sweep, for 100 held-out evaluations in total.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)
print('Python executable:', sys.executable)

In [ ]:
import importlib.metadata
import importlib.util
import signal
import subprocess

requirements_path = PROJECT_DIR / 'requirements.txt'
installed_torch = importlib.metadata.version('torch') if importlib.util.find_spec('torch') else ''
target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(requirements_path)])
if not target_ready:
    print('Pinned Torch/CUDA stack installed. Restarting the runtime; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')

In [ ]:
import torch
import torch_geometric
import torch_scatter
import torch_sparse

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch:', torch.__version__)
subprocess.run([sys.executable, 'scripts/run_graph_classification_repeated_cv_wandb_tuning.py', '--help'], check=True)

In [ ]:
DATASET = 'mutag'  # mutag, proteins, enzymes, or nci1
BASE_CONFIG_PATH = Path(f'configs/experiments/{DATASET}_h2snn.yaml')
EXP_NAME = f'{DATASET}_h2snn_repeated_10x10_kfold'
OUTPUT_DIR = Path('experiments/colab')
STATE_DIR = OUTPUT_DIR / EXP_NAME / 'tuning_state'

NUM_REPETITIONS = 5
NUM_FOLDS = 10
TOTAL_UNITS = NUM_REPETITIONS * NUM_FOLDS
UNIT_START = 0
UNIT_COUNT = TOTAL_UNITS
BASE_SEED = 42
SPLIT_SEED = 42
N_TRIALS = 30
EPOCHS = 500
PATIENCE = 50
HYPERBAND_MIN_ITER = 30
HYPERBAND_ETA = 3
D_VALUES = [16, 32, 64]
MONITOR = 'auroc'

RESUME = False
SWEEP_ID = ''  # only for RESUME=True with UNIT_COUNT=1
WANDB_PROJECT = 'h2snn'
WANDB_ENTITY = ''
WANDB_MODE = 'online'
WANDB_TAGS = ['paper-quality', 'graph-classification', 'repeated-kfold']
print('Estimated model fits:', TOTAL_UNITS * N_TRIALS)

In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None
wandb.login(key=wandb_key, relogin=True) if wandb_key else wandb.login()

In [ ]:
cmd = [
    sys.executable, 'scripts/run_graph_classification_repeated_cv_wandb_tuning.py',
    '--config', str(BASE_CONFIG_PATH), '--exp_name', EXP_NAME,
    '--output_dir', str(OUTPUT_DIR), '--device', DEVICE,
    '--num_repetitions', str(NUM_REPETITIONS), '--num_folds', str(NUM_FOLDS),
    '--unit_start', str(UNIT_START), '--unit_count', str(UNIT_COUNT),
    '--base_seed', str(BASE_SEED), '--split_seed', str(SPLIT_SEED),
    '--n_trials', str(N_TRIALS), '--epochs', str(EPOCHS),
    '--patience', str(PATIENCE), '--monitor', MONITOR,
    '--hyperband_min_iter', str(HYPERBAND_MIN_ITER), '--hyperband_eta', str(HYPERBAND_ETA),
    '--state_dir', str(STATE_DIR), '--wandb_project', WANDB_PROJECT,
    '--wandb_mode', WANDB_MODE, '--wandb_tags', *WANDB_TAGS,
    '--d_values', *[str(value) for value in D_VALUES],
]
if WANDB_ENTITY:
    cmd.extend(['--wandb_entity', WANDB_ENTITY])
if RESUME:
    cmd.append('--resume')
if SWEEP_ID:
    cmd.extend(['--sweep_id', SWEEP_ID])
print('Validating:', ' '.join([*cmd, '--validate_only']))
subprocess.run([*cmd, '--validate_only'], check=True)
print('Starting:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json

summary_path = OUTPUT_DIR / EXP_NAME / 'repeated_cv_tuning_summary.json'
if summary_path.exists():
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(payload['summary'], indent=2)[:12000])
else:
    print('Summary not complete yet. Fold states:')
    for path in sorted(STATE_DIR.glob('tuning_state_rep*_fold*.json')):
        state = json.loads(path.read_text(encoding='utf-8'))
        print(path.name, state.get('completed_trials', 0), '/', state.get('target_trials', N_TRIALS))